# Deep Learning Project Work

In this project, I will explore the benchmarking of DINO models for an image classification task. The models to be evaluated include:

- (?) DINOv2 (facebook/dinov2) (?)
- DINOv2 base with registers (facebook/dinov2-with-registers-base)
- DINOv3 base (facebook/dinov3-vitb16-pretrain-lvd1689m)

The dataset on which the models will be trained and evaluated is the dataset available at the [link](https://zenodo.org/records/10137731) relative to [this paper](https://onlinelibrary.wiley.com/doi/10.1111/gcb.17078), regarding micrometeorological conditions across 49 wildlife cameras in South Africa's Maloti-Drakensberg and the Swiss Alps, with classifications for **overcast**, **sunshine**, **hail**, and **snow** conditions.

In the original paper, 

In [3]:
import numpy as np
import pandas as pd
import random
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
import ipywidgets as widgets
from ipywidgets import interact

SEED=42

DATASET_PATH = "/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_weather_ct/"

IMAGES_METADATA_CSV = DATASET_PATH + "image_metadata.csv"

IMAGES_DIR = DATASET_PATH + "images/"

CLASS_NAMES = [
    "background",
    "sunshine",
    "snow",
    "hail",
]

def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

## Analyzing dataset structure

Attached to the dataset there are CSV files with metadata for images and cameras. The main file for images is `image_data.csv`, with these columns, not all of which are needed/useful for model training:

- `DatasetOutFileName`: image file name.
- `label_sample`: sampling strategy used to select the image (not needed for model training).
- `Region`: camera region (e.g. Maloti-Drakensberg `ZA` or Swiss Alps `CH`).
- `cam`: unique camera ID that captured the image.
- `DateTimeOriginal`: original date/time from image metadata (raw format, may contain colons).
- `DateTime`: reformatted date/time (standardized representation).
- `DOY`: day of year ($1–366$).
- `Hour`: hour of capture ($0–23$).
- `Minute`: minute of capture ($0–59$).
- `Temperature`: ambient temperature recorded by the sensor at capture time.
- `Flash`: whether flash fired ($1$) or not ($0$).
- `ExposureTime`: shutter exposure time in seconds (decimal).
- `weather`: numeric weather class label (target for the model, e.g. 0–3).
- `weather_desc`: textual weather class (e.g. `background`, `sunshine`, `hail`, `snow`).
- `cis_fold`: fold index ($0–4$) for **Cis fivefold cross‑validation**, a within‑domain, image‑level random split where images from all cameras are randomly assigned to folds. Folds are mutually exclusive, stratified by the four study sites, and each fold is iteratively used as validation while the other four form the training set.
- `trans_fold`: fold index ($0–4$) for **Trans fivefold cross‑validation**, a camera‑level split where entire cameras (all their images) are assigned to folds so that no camera appears in both training and validation. Folds are mutually exclusive and stratified by the four study sites.
- `holdout`: boolean indicator (`TRUE` if reserved exclusively for the final test set).
- `cis_prod_fold`: alternative Cis fold variant (also a fivefold, site‑stratified Cis split) used for production/alternate experiment splits or final model selection; similar semantics to `cis_fold` but intended for deployment/production evaluation.


In [4]:
import os
from collections import defaultdict

df_metadata_images = pd.read_csv(IMAGES_METADATA_CSV)
df_metadata_images.head()

,DatasetOutFileName,label_sample,Region,cam,DateTimeOriginal,DateTime,DOY,Hour,Minute,Temperature,Flash,ExposureTime,weather,weather_desc,cis_fold,trans_fold,holdout,cis_prod_fold
0,000001.JPG,systematic,ZA,SA16,2022:01:12 21:01:12,12/01/2022 21:01,12,21,1,10,1,0.065986,0,background,2,4,False,4
1,000002.JPG,strategic,ZA,SA07,2021:12:25 03:03:27,25/12/2021 03:03,359,3,3,2,1,0.065986,3,hail,4,2,False,1
2,000003.JPG,systematic,ZA,SA06,2022:04:01 15:00:27,01/04/2022 15:00,91,15,0,6,0,0.004672,1,sunshine,1,1,False,1
3,000004.JPG,strategic,CH,C02,2021:09:04 01:00:01,04/09/2021 01:00,247,1,0,7,1,0.065986,0,background,6,6,True,5
4,000005.JPG,systematic,CH,C17,2021:08:31 09:01:31,31/08/2021 09:01,243,9,1,5,0,0.002625,1,sunshine,2,3,False,5


## Load dataset

Because the dataset identifies images using only their file names (without paths), and images are organized in subdirectories by class ID, a function is needed to resolve the full image paths based on the `DatasetOutFileName` and the directory structure.

In [5]:
def resolve_filename(fname: str) -> str:
    hits = file_map.get(fname)
    if not hits:
        return None
    if len(hits) > 1:
        # even if should not happen, warn if multiple matches, default to first
        print(f"Warning: multiple matches for {fname}; using first match: {hits[0]}")
    return hits[0]

file_map = defaultdict(list)
for root, _, files in os.walk(IMAGES_DIR):
    for fname in files:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            continue
        file_map[fname].append(os.path.abspath(os.path.join(root, fname)))

df_metadata_images['DatasetAbsPath'] = df_metadata_images['DatasetOutFileName'].map(resolve_filename)

missing = df_metadata_images['DatasetAbsPath'].isna().sum()
print(f"Resolved paths for {len(df_metadata_images) - missing}/{len(df_metadata_images)} images. Missing: {missing}")

print(df_metadata_images['weather_desc'].value_counts())
display(df_metadata_images[['DatasetOutFileName', 'DatasetAbsPath']].head())

Resolved paths for 8953/8953 images. Missing: 0
weather_desc
background    5492
sunshine      1586
snow          1033
hail           842
Name: count, dtype: int64


,DatasetOutFileName,DatasetAbsPath
0,000001.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
1,000002.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
2,000003.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
3,000004.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...
4,000005.JPG,/mnt/synas/ngb/AI_DATASETS/zenodo_10137731_wea...


In [17]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class WeatherDataset(Dataset):
    def __init__(self, metadata, processor, augmentations=None, root_dir=""):
        self.metadata = metadata.reset_index(drop=True)
        self.image_paths = self.metadata['DatasetAbsPath'].tolist() 
        self.labels = self.metadata['weather'].tolist()
        self.processor = processor
        self.augmentations = augmentations
        self.root_dir = root_dir
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        rel_path = self.image_paths[idx]
        image_path = os.path.join(self.root_dir, rel_path)
        
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            return self.__getitem__(np.random.randint(0, len(self)))

        if self.augmentations:
            image = self.augmentations(image)
        encoding = self.processor(image, return_tensors="pt")
        
        return {
            'pixel_values': encoding['pixel_values'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

## Utils functions for splits creation

In [16]:
from typing import Tuple

def is_holdout(df: pd.DataFrame, holdout_col: str = 'holdout') -> pd.Series:
    '''
    Function to determine holdout rows in DataFrame.
    Inputs:
    - df: input DataFrame
    Outputs:
    - pd.Series of booleans indicating holdout status
    '''
    s = df.get('holdout')
    if s is None:
        return pd.Series([False] * len(df), index=df.index)
    if s.dtype == bool:
        return s.fillna(False)
    s_str = s.astype(str).str.upper()
    return s_str == 'TRUE'

def make_splits(df: pd.DataFrame, fold_col: str, fold_index: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''
    Function to split DataFrame into train/val/test sets according to specified fold column and index.
    Inputs:
    - df: input DataFrame
    - fold_col: name of the column containing fold assignments
    - fold_index: index of the fold to use as validation set
    Outputs:
    - train_df: training set DataFrame
    - val_df: validation set DataFrame
    - test_df: test set DataFrame
    '''
    df_cp = df.copy()

    holdout_mask = is_holdout(df_cp)
    test_df = df_cp[holdout_mask].reset_index(drop=True)
    df_no_holdout = df_cp[~holdout_mask].reset_index(drop=True)

    if fold_col not in df_no_holdout.columns:
        raise KeyError(f'Fold column {fold_col} not in DataFrame columns: {list(df_no_holdout.columns)}')

    val_mask = df_no_holdout[fold_col] == fold_index
    val_df = df_no_holdout[val_mask].reset_index(drop=True)
    train_df = df_no_holdout[~val_mask].reset_index(drop=True)

    return train_df, val_df, test_df

def make_cis_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'cis_fold', fold_index)

""" def make_trans_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'trans_fold', fold_index)

def make_cis_prod_splits(df: pd.DataFrame, fold_index: int = 0):
    return make_splits(df, 'cis_prod_fold', fold_index) """

" def make_trans_splits(df: pd.DataFrame, fold_index: int = 0):\n    return make_splits(df, 'trans_fold', fold_index)\n\ndef make_cis_prod_splits(df: pd.DataFrame, fold_index: int = 0):\n    return make_splits(df, 'cis_prod_fold', fold_index) "

## Visualizing full dataset

In [18]:
import matplotlib.pyplot as plt

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")

dataset = WeatherDataset(
    metadata=df_metadata_images,
    processor=processor,
    root_dir=IMAGES_DIR
)

@interact(
    sample_index = widgets.IntSlider(min=0, max=len(dataset)-1, step=1, value=42)
)
def plot_image(sample_index: int) -> None:
    sample = dataset[sample_index]
    pv = sample['pixel_values']

    img = pv.permute(1, 2, 0).numpy()

    mean = np.array(processor.image_mean)
    std = np.array(processor.image_std)

    img = (img * std + mean)
    img = (img * 255.0).clip(0, 255).astype("uint8")

    label_idx = sample['labels'].item()
    label_str = CLASS_NAMES[label_idx] if 'CLASS_NAMES' in globals() else str(label_idx)
    
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Label: {label_str}")
    plt.axis('off')
    plt.show()

interactive(children=(IntSlider(value=42, description='sample_index', max=8952), Output()), _dom_classes=('wid…

## Training and evaluating DINO models

### DINOv2(w/o registers)

Parlare di augmentations usate, hyperparametri, ecc.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import pandas as pd
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments
import torchvision.transforms as T
import evaluate
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

dinov2_metrics = []

class_names = ["background", "sunshine", "snow", "hail"] 
id2label = {0: "background", 1: "sunshine", 2: "snow", 3: "hail"}
label2id = {v: k for k, v in id2label.items()}

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")

for fold_idx in range(5):
    print(f"\n{'='*20} STARTING FOLD {fold_idx} {'='*20}")

    train_df, val_df, test_df = make_cis_splits(df_metadata_images, fold_index=fold_idx)
    
    transforms_train = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    ])

    # PASSA IL PROCESSOR AL DATASET!
    train_ds = WeatherDataset(train_df, processor=processor, augmentations=transforms_train)
    val_ds = WeatherDataset(val_df, processor=processor)
    test_ds = WeatherDataset(test_df, processor=processor)

    model = AutoModelForImageClassification.from_pretrained(
        "facebook/dinov2-base",
        num_labels=len(class_names),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
        cache_dir="models-cache"
    )
    
    # Congelamento backbone (Ottimo per velocità)
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

    args = TrainingArguments(
        output_dir=f"./dinov2_results/fold_{fold_idx}",
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-4, # Aumentato leggermente LR dato che alleni solo la testa
        per_device_train_batch_size=64, # Se da OOM, scendi a 32
        per_device_eval_batch_size=64,
        num_train_epochs=10,
        warmup_ratio=0.1,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=8, # 8 è spesso il "sweet spot", 16 a volte crea overhead eccessivo
        dataloader_pin_memory=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        # data_collator RIMOSSO -> Usa quello default che è molto più veloce per tensori
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    print(f"\nGenerating Classification Report for Fold {fold_idx}...")
    
    pred_output = trainer.predict(test_ds)
    
    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=1)
    
    print(f"\n--- Classification Report Fold {fold_idx} ---")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    
    results = {
        "fold": fold_idx,
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }
    
    dinov2_metrics.append(results)
    print(f"Metrics saved for Fold {fold_idx}: {results}\n")
    
    del model
    del trainer
    torch.cuda.empty_cache()

df_results = pd.DataFrame(dinov2_metrics)
print("\n=== FINAL RESULTS ACROSS ALL FOLDS ===")
print(df_results)
print("\nAverage Metrics:")
print(df_results.mean(numeric_only=True))


==================== STARTING FOLD 0 ====================


Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
